In [35]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import confusion_matrix
from sklearn.metrics import RocCurveDisplay

In [19]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 2000

# Предикторы
tariff_type = np.random.choice([0, 1, 2], size=n, p=[0.4, 0.3, 0.3])
age = np.random.randint(18, 76, size=n)
watch_hours_week = np.round(np.random.uniform(0, 40, size=n), 1)
content_completion = np.round(np.random.uniform(0.0, 1.0, size=n), 2)
device_type = np.random.choice([0, 1, 2], size=n, p=[0.4, 0.35, 0.25])
promo_used = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
support_tickets = np.random.poisson(lam=1.5, size=n)
support_tickets = np.clip(support_tickets, 0, 15)
friend_referrals = np.random.choice([0, 1], size=n, p=[0.8, 0.2])

# Таргет с разными зависимостями для каждого tariff_type
subscription_renew = np.zeros(n, dtype=int)

for i in range(n):
    if tariff_type[i] == 0:  # базовый
        prob = 0.3 + 0.3 * promo_used[i] - 0.05 * support_tickets[i] + 0.1 * friend_referrals[i]
        prob = np.clip(prob, 0.05, 0.95)
    elif tariff_type[i] == 1:  # премиум
        prob = 0.2 + 0.4 * (watch_hours_week[i] / 40) + 0.2 * (device_type[i] == 2) + 0.1 * content_completion[i]
        prob = np.clip(prob, 0.05, 0.95)
    else:  # студенческий
        prob = 0.1 + 0.3 * (1 - (age[i] - 18) / 57) + 0.3 * friend_referrals[i] + 0.1 * content_completion[i]
        prob = np.clip(prob, 0.05, 0.95)

    subscription_renew[i] = np.random.binomial(1, prob)

df1 = pd.DataFrame({
    'tariff_type': tariff_type,
    'age': age,
    'watch_hours_week': watch_hours_week,
    'content_completion': content_completion,
    'device_type': device_type,
    'promo_used': promo_used,
    'support_tickets': support_tickets,
    'friend_referrals': friend_referrals,
    'subscription_renew': subscription_renew
})

print(df1.head())
print(f"Соотношение таргета: {df1['subscription_renew'].mean():.2f}")


   tariff_type  age  watch_hours_week  content_completion  device_type  \
0            0   45              36.6                0.57            2   
1            2   24              17.4                0.33            1   
2            2   52              10.3                0.32            0   
3            1   72              17.4                0.11            0   
4            0   45              28.9                0.37            0   

   promo_used  support_tickets  friend_referrals  subscription_renew  
0           0                2                 0                   1  
1           1                0                 0                   0  
2           0                2                 0                   1  
3           0                2                 0                   0  
4           0                0                 0                   0  
Соотношение таргета: 0.40


In [21]:
df1['subscription_renew'].value_counts()

0    1205
1     795
Name: subscription_renew, dtype: int64

In [22]:
import pandas as pd
import numpy as np

np.random.seed(123)
n = 2000

# Предикторы
membership_type = np.random.choice([0, 1, 2], size=n, p=[0.25, 0.35, 0.4])
distance_km = np.round(np.random.uniform(0.5, 20.0, size=n), 1)
visits_per_week = np.round(np.random.uniform(0, 7, size=n), 1)
duration_months = np.random.randint(1, 61, size=n)
age = np.random.randint(18, 66, size=n)
gender = np.random.choice([0, 1], size=n, p=[0.5, 0.5])
personal_goal = np.random.choice([0, 1], size=n, p=[0.4, 0.6])
late_cancels = np.random.poisson(lam=0.8, size=n)
late_cancels = np.clip(late_cancels, 0, 10)

# Таргет с разными зависимостями для каждого membership_type
churn = np.zeros(n, dtype=int)

for i in range(n):
    if membership_type[i] == 0:  # индивидуальные
        prob = 0.1 + 0.1 * late_cancels[i] - 0.2 * personal_goal[i] + 0.05 * distance_km[i]
        prob = np.clip(prob, 0.05, 0.95)
    elif membership_type[i] == 1:  # групповые
        prob = 0.15 - 0.15 * (gender[i] == 1) - 0.05 * (age[i] / 65) + 0.1 * late_cancels[i]
        prob = np.clip(prob, 0.05, 0.95)
    else:  # свободное посещение
        prob = 0.2 + 0.03 * distance_km[i] - 0.1 * visits_per_week[i] + 0.05 * (6 - duration_months[i] / 10)
        prob = np.clip(prob, 0.05, 0.95)

    churn[i] = np.random.binomial(1, prob)

df2 = pd.DataFrame({
    'membership_type': membership_type,
    'distance_km': distance_km,
    'visits_per_week': visits_per_week,
    'duration_months': duration_months,
    'age': age,
    'gender': gender,
    'personal_goal': personal_goal,
    'late_cancels': late_cancels,
    'churn': churn
})

print(df2.head())
print(f"Соотношение таргета: {df2['churn'].mean():.2f}")


   membership_type  distance_km  visits_per_week  duration_months  age  \
0                2          8.7              1.1               22   65   
1                1          8.2              6.8                9   56   
2                0          1.2              4.3                9   60   
3                1         17.7              1.1               37   24   
4                2         13.7              5.6                4   62   

   gender  personal_goal  late_cancels  churn  
0       0              1             1      0  
1       1              0             1      0  
2       1              0             0      0  
3       1              0             0      0  
4       0              1             1      0  
Соотношение таргета: 0.34


In [24]:
df2['churn'].value_counts()

0    1330
1     670
Name: churn, dtype: int64

In [79]:
# df1.to_csv('data/kinomir.csv', index = False)

In [26]:
# df2.to_csv('data/fit_zone.csv', index=False)

In [27]:
df2

,membership_type,distance_km,visits_per_week,duration_months,age,gender,personal_goal,late_cancels,churn
0,2,8.7,1.1,22,65,0,1,1,0
1,1,8.2,6.8,9,56,1,0,1,0
2,0,1.2,4.3,9,60,1,0,0,0
3,1,17.7,1.1,37,24,1,0,0,0
4,2,13.7,5.6,4,62,0,1,1,0
...,...,...,...,...,...,...,...,...,...
1995,0,11.3,7.0,27,24,0,1,1,1
1996,2,11.1,5.3,9,47,0,0,2,0
1997,1,14.9,3.4,56,21,0,1,2,1
1998,2,14.2,4.7,8,62,0,1,1,0


In [36]:
import pandas as pd
import numpy as np

np.random.seed(123)
n = 2000

# Предикторы
membership_type = np.random.choice([0, 1, 2], size=n, p=[0.25, 0.35, 0.4])
distance_km = np.round(np.random.uniform(0.5, 20.0, size=n), 1)
visits_per_week = np.round(np.random.uniform(0, 7, size=n), 1)
duration_months = np.random.randint(1, 61, size=n)
age = np.random.randint(18, 66, size=n)
gender = np.random.choice([0, 1], size=n, p=[0.5, 0.5])
personal_goal = np.random.choice([0, 1], size=n, p=[0.4, 0.6])
late_cancels = np.random.poisson(lam=0.8, size=n)
late_cancels = np.clip(late_cancels, 0, 10)

# Таргет с разными зависимостями для каждого membership_type
churn = np.zeros(n, dtype=int)

for i in range(n):
    # индивидуальные
    if membership_type[i] == 0 and personal_goal[i] == 1 and late_cancels[i] == 1:
        churn[i] = 1
    # групповые
    elif membership_type[i] == 1 and personal_goal[i] == 0 and late_cancels[i] == 0:
        churn[i] = 1
    # свободное посещение
    elif membership_type[i] == 2 and gender[i] == 0 and late_cancels[i] == 1:
        churn[i] = 1 


df2 = pd.DataFrame({
    'membership_type': membership_type,
    'distance_km': distance_km,
    'visits_per_week': visits_per_week,
    'duration_months': duration_months,
    'age': age,
    'gender': gender,
    'personal_goal': personal_goal,
    'late_cancels': late_cancels,
    'churn': churn
})

print(df2.head())
print(f"Соотношение таргета: {df2['churn'].mean():.2f}")


   membership_type  distance_km  visits_per_week  duration_months  age  \
0                2          8.7              1.1               22   65   
1                1          8.2              6.8                9   56   
2                0          1.2              4.3                9   60   
3                1         17.7              1.1               37   24   
4                2         13.7              5.6                4   62   

   gender  personal_goal  late_cancels  churn  
0       0              1             1      1  
1       1              0             1      0  
2       1              0             0      0  
3       1              0             0      1  
4       0              1             1      1  
Соотношение таргета: 0.20


In [60]:
import pandas as pd
import numpy as np

np.random.seed(123)
n = 2000

# Предикторы
membership_type = np.random.choice([0, 1, 2], size=n, p=[0.25, 0.35, 0.4])
distance_km = np.round(np.random.uniform(0.5, 20.0, size=n), 1)
visits_per_week = np.round(np.random.uniform(0, 7, size=n), 1)
duration_months = np.random.randint(1, 61, size=n)
age = np.random.randint(18, 66, size=n)
gender = np.random.choice([0, 1], size=n, p=[0.5, 0.5])
personal_goal = np.random.choice([0, 1], size=n, p=[0.4, 0.6])
late_cancels = np.random.poisson(lam=0.8, size=n)
late_cancels = np.clip(late_cancels, 0, 10)

# Таргет с разными зависимостями для каждого membership_type
churn = np.zeros(n, dtype=int)

for i in range(n):
    if membership_type[i] == 0:  # индивидуальные
        prob = 0.1 + 0.1 * late_cancels[i] - 0.2 * personal_goal[i] + 0.05 * distance_km[i]
        prob = np.clip(prob, 0.0, 1.0)
    elif membership_type[i] == 1:  # групповые
        prob = 0.15 - 0.15 * (gender[i] == 1) - 0.05 * (age[i] / 65) + 0.1 * late_cancels[i]
        prob = np.clip(prob, 0.0, 1.0)
    else:  # свободное посещение
        prob = 0.2 + 0.03 * distance_km[i] - 0.1 * visits_per_week[i] + 0.05 * (6 - duration_months[i] / 10)
        prob = np.clip(prob, 0.00, 1.0)

    churn[i] = np.random.binomial(1, prob)

df2 = pd.DataFrame({
    'membership_type': membership_type,
    'distance_km': distance_km,
    'visits_per_week': visits_per_week,
    'duration_months': duration_months,
    'age': age,
    'gender': gender,
    'personal_goal': personal_goal,
    'late_cancels': late_cancels,
    'churn': churn
})

# print(df2.head())
# print(f"Соотношение таргета: {df2['churn'].mean():.2f}")


df = df2.copy()

X = df.drop(columns = ['churn'])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = KNeighborsClassifier()
model.fit(X_train, y_train)

y_test_pred = model.predict(X_test)

print('')
print('До локализаций')
print('')
accuracy = accuracy_score(y_test, y_test_pred)
print('accuracy =', accuracy)
precision = precision_score(y_test, y_test_pred)
print('precision =', precision)
recall = recall_score(y_test, y_test_pred)
print('recall =', recall)
f1 = f1_score(y_test, y_test_pred)
print('f1 =', f1)

X = df.drop(columns = ['churn'])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train_0 = X_train.loc[X_train['membership_type'] == 0]
y_train_0 = y_train.loc[X_train_0.index]
model_0 = KNeighborsClassifier()
model_0.fit(X_train_0, y_train_0)

X_train_1 = X_train.loc[X_train['membership_type'] == 1]
y_train_1 = y_train.loc[X_train_1.index]
model_1 = KNeighborsClassifier()
model_1.fit(X_train_1, y_train_1)

X_train_2 = X_train.loc[X_train['membership_type'] == 2]
y_train_2 = y_train.loc[X_train_2.index]
model_2 = KNeighborsClassifier()
model_2.fit(X_train_2, y_train_2)

print('')
print('')
print('Частные скоры на классах')
print('')
X_test_0 = X_test.loc[X_test['membership_type'] == 0]
y_test_0 = y_test.loc[X_test_0.index]
y_test_pred_0 = model_0.predict(X_test_0)
print('accuracy_0 =', accuracy_score(y_test_0, y_test_pred_0))
print('precision_0 =', precision_score(y_test_0, y_test_pred_0))
print('recall_0 =', recall_score(y_test_0, y_test_pred_0))
print('f1_0 =', f1_score(y_test_0, y_test_pred_0))
print('')
X_test_1 = X_test.loc[X_test['membership_type'] == 1]
y_test_1 = y_test.loc[X_test_1.index]
y_test_pred_1 = model_1.predict(X_test_1)
print('accuracy_1 =', accuracy_score(y_test_1, y_test_pred_1))
print('precision_1 =', precision_score(y_test_1, y_test_pred_1))
print('recall_1 =', recall_score(y_test_1, y_test_pred_1))
print('f1_1 =', f1_score(y_test_1, y_test_pred_1))
print('')
X_test_2 = X_test.loc[X_test['membership_type'] == 2]
y_test_2 = y_test.loc[X_test_2.index]
y_test_pred_2 = model_2.predict(X_test_2)
print('accuracy_2 =', accuracy_score(y_test_2, y_test_pred_2))
print('precision_2 =', precision_score(y_test_2, y_test_pred_2))
print('recall_2 =', recall_score(y_test_2, y_test_pred_2))
print('f1_2 =', f1_score(y_test_2, y_test_pred_2))

df_y_test_y_pred_0 = pd.DataFrame()
df_y_test_y_pred_0['y_test'] = y_test_0
df_y_test_y_pred_0['y_test_pred'] = y_test_pred_0
df_y_test_y_pred_0

df_y_test_y_pred_1 = pd.DataFrame()
df_y_test_y_pred_1['y_test'] = y_test_1
df_y_test_y_pred_1['y_test_pred'] = y_test_pred_1
df_y_test_y_pred_1

df_y_test_y_pred_2 = pd.DataFrame()
df_y_test_y_pred_2['y_test'] = y_test_2
df_y_test_y_pred_2['y_test_pred'] = y_test_pred_2
df_y_test_y_pred_2


df_y_test_y_pred = pd.concat([df_y_test_y_pred_0, df_y_test_y_pred_1, df_y_test_y_pred_2])
df_y_test_y_pred

print('')
print('')
print('Общие скоры после локализаций')
print('')
accuracy = accuracy_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('accuracy = ', accuracy)
precision = precision_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('precision = ', precision)
recall = recall_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('recall = ', recall)
f1 = f1_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('f1 = ', f1)


До локализаций

accuracy = 0.642
precision = 0.41353383458646614
recall = 0.3525641025641026
f1 = 0.3806228373702422


Частные скоры на классах

accuracy_0 = 0.6910569105691057
precision_0 = 0.6883116883116883
recall_0 = 0.7910447761194029
f1_0 = 0.7361111111111112

accuracy_1 = 0.8488372093023255
precision_1 = 0.3333333333333333
recall_1 = 0.08333333333333333
f1_1 = 0.13333333333333333

accuracy_2 = 0.6682926829268293
precision_2 = 0.47619047619047616
recall_2 = 0.46153846153846156
f1_2 = 0.46875


Общие скоры после локализаций

accuracy =  0.736
precision =  0.5821917808219178
recall =  0.5448717948717948
f1 =  0.5629139072847682


In [62]:
df2

,membership_type,distance_km,visits_per_week,duration_months,age,gender,personal_goal,late_cancels,churn
0,2,8.7,1.1,22,65,0,1,1,1
1,1,8.2,6.8,9,56,1,0,1,0
2,0,1.2,4.3,9,60,1,0,0,0
3,1,17.7,1.1,37,24,1,0,0,1
4,2,13.7,5.6,4,62,0,1,1,1
...,...,...,...,...,...,...,...,...,...
1995,0,11.3,7.0,27,24,0,1,1,1
1996,2,11.1,5.3,9,47,0,0,2,0
1997,1,14.9,3.4,56,21,0,1,2,0
1998,2,14.2,4.7,8,62,0,1,1,1


In [66]:
import pandas as pd
import numpy as np

np.random.seed(123)
n = 2000

# Предикторы
membership_type = np.random.choice([0, 1, 2], size=n, p=[0.25, 0.35, 0.4])
distance_km = np.round(np.random.uniform(0.5, 20.0, size=n), 1)
visits_per_week = np.round(np.random.uniform(0, 7, size=n), 1)
duration_months = np.random.randint(1, 61, size=n)
age = np.random.randint(18, 66, size=n)
gender = np.random.choice([0, 1], size=n, p=[0.5, 0.5])
personal_goal = np.random.choice([0, 1], size=n, p=[0.4, 0.6])
late_cancels = np.random.poisson(lam=0.8, size=n)
late_cancels = np.clip(late_cancels, 0, 10)

# Таргет с разными зависимостями для каждого membership_type
churn = np.zeros(n, dtype=int)

for i in range(n):
    # индивидуальные
    if membership_type[i] == 0 and duration_months[i] >20  and age[i] >45:
        churn[i] = 1
    # групповые
    elif membership_type[i] == 1 and distance_km[i] > 15 and age[i] > 45:
        churn[i] = 1
    # свободное посещение
    elif membership_type[i] == 2 and duration_months[i] > 20 and distance_km[i] > 15:
        churn[i] = 1 

df2 = pd.DataFrame({
    'membership_type': membership_type,
    'distance_km': distance_km,
    'visits_per_week': visits_per_week,
    'duration_months': duration_months,
    'age': age,
    'gender': gender,
    'personal_goal': personal_goal,
    'late_cancels': late_cancels,
    'churn': churn
})

# print(df2.head())
# print(f"Соотношение таргета: {df2['churn'].mean():.2f}")


df = df2.copy()

X = df.drop(columns = ['churn'])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = KNeighborsClassifier()
model.fit(X_train, y_train)

y_test_pred = model.predict(X_test)

print('')
print('До локализаций')
print('')
accuracy = accuracy_score(y_test, y_test_pred)
print('accuracy =', accuracy)
precision = precision_score(y_test, y_test_pred)
print('precision =', precision)
recall = recall_score(y_test, y_test_pred)
print('recall =', recall)
f1 = f1_score(y_test, y_test_pred)
print('f1 =', f1)

X = df.drop(columns = ['churn'])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train_0 = X_train.loc[X_train['membership_type'] == 0]
y_train_0 = y_train.loc[X_train_0.index]
model_0 = KNeighborsClassifier()
model_0.fit(X_train_0, y_train_0)

X_train_1 = X_train.loc[X_train['membership_type'] == 1]
y_train_1 = y_train.loc[X_train_1.index]
model_1 = KNeighborsClassifier()
model_1.fit(X_train_1, y_train_1)

X_train_2 = X_train.loc[X_train['membership_type'] == 2]
y_train_2 = y_train.loc[X_train_2.index]
model_2 = KNeighborsClassifier()
model_2.fit(X_train_2, y_train_2)

print('')
print('')
print('Частные скоры на классах')
print('')
X_test_0 = X_test.loc[X_test['membership_type'] == 0]
y_test_0 = y_test.loc[X_test_0.index]
y_test_pred_0 = model_0.predict(X_test_0)
print('accuracy_0 =', accuracy_score(y_test_0, y_test_pred_0))
print('precision_0 =', precision_score(y_test_0, y_test_pred_0))
print('recall_0 =', recall_score(y_test_0, y_test_pred_0))
print('f1_0 =', f1_score(y_test_0, y_test_pred_0))
print('')
X_test_1 = X_test.loc[X_test['membership_type'] == 1]
y_test_1 = y_test.loc[X_test_1.index]
y_test_pred_1 = model_1.predict(X_test_1)
print('accuracy_1 =', accuracy_score(y_test_1, y_test_pred_1))
print('precision_1 =', precision_score(y_test_1, y_test_pred_1))
print('recall_1 =', recall_score(y_test_1, y_test_pred_1))
print('f1_1 =', f1_score(y_test_1, y_test_pred_1))
print('')
X_test_2 = X_test.loc[X_test['membership_type'] == 2]
y_test_2 = y_test.loc[X_test_2.index]
y_test_pred_2 = model_2.predict(X_test_2)
print('accuracy_2 =', accuracy_score(y_test_2, y_test_pred_2))
print('precision_2 =', precision_score(y_test_2, y_test_pred_2))
print('recall_2 =', recall_score(y_test_2, y_test_pred_2))
print('f1_2 =', f1_score(y_test_2, y_test_pred_2))

df_y_test_y_pred_0 = pd.DataFrame()
df_y_test_y_pred_0['y_test'] = y_test_0
df_y_test_y_pred_0['y_test_pred'] = y_test_pred_0
df_y_test_y_pred_0

df_y_test_y_pred_1 = pd.DataFrame()
df_y_test_y_pred_1['y_test'] = y_test_1
df_y_test_y_pred_1['y_test_pred'] = y_test_pred_1
df_y_test_y_pred_1

df_y_test_y_pred_2 = pd.DataFrame()
df_y_test_y_pred_2['y_test'] = y_test_2
df_y_test_y_pred_2['y_test_pred'] = y_test_pred_2
df_y_test_y_pred_2


df_y_test_y_pred = pd.concat([df_y_test_y_pred_0, df_y_test_y_pred_1, df_y_test_y_pred_2])
df_y_test_y_pred

print('')
print('')
print('Общие скоры после локализаций')
print('')
accuracy = accuracy_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('accuracy = ', accuracy)
precision = precision_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('precision = ', precision)
recall = recall_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('recall = ', recall)
f1 = f1_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('f1 = ', f1)


До локализаций

accuracy = 0.878
precision = 0.7230769230769231
recall = 0.5222222222222223
f1 = 0.6064516129032258


Частные скоры на классах

accuracy_0 = 0.959349593495935
precision_0 = 1.0
recall_0 = 0.8571428571428571
f1_0 = 0.923076923076923

accuracy_1 = 0.9534883720930233
precision_1 = 0.8
recall_1 = 0.7058823529411765
f1_1 = 0.7500000000000001

accuracy_2 = 0.9463414634146341
precision_2 = 0.8461538461538461
recall_2 = 0.868421052631579
f1_2 = 0.8571428571428572


Общие скоры после локализаций

accuracy =  0.952
precision =  0.8928571428571429
recall =  0.8333333333333334
f1 =  0.8620689655172413


In [65]:
df2['churn'].mean()

0.1785

In [67]:
# df2.to_csv('data/fit_zone.csv', index=False)

In [78]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 2000

# Предикторы
tariff_type = np.random.choice([0, 1, 2], size=n, p=[0.4, 0.3, 0.3])
age = np.random.randint(18, 76, size=n)
watch_hours_week = np.round(np.random.uniform(0, 40, size=n), 1)
content_completion = np.round(np.random.uniform(0.0, 1.0, size=n), 2)
device_type = np.random.choice([0, 1, 2], size=n, p=[0.4, 0.35, 0.25])
promo_used = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
support_tickets = np.random.poisson(lam=1.5, size=n)
support_tickets = np.clip(support_tickets, 0, 15)
friend_referrals = np.random.choice([0, 1], size=n, p=[0.8, 0.2])

# Таргет с разными зависимостями для каждого tariff_type
subscription_renew = np.zeros(n, dtype=int)

for i in range(n):
    # индивидуальные
    if tariff_type[i] == 0 and age[i] > 45  and watch_hours_week[i] > 20:
        subscription_renew[i] = 1
    # групповые
    elif tariff_type[i] == 1 and age[i] > 45 and watch_hours_week[i] < 20:
        subscription_renew[i] = 1
    # свободное посещение
    elif tariff_type[i] == 2 and age[i] < 40 and watch_hours_week[i] > 20:
        subscription_renew[i] = 1 
df1 = pd.DataFrame({
    'tariff_type': tariff_type,
    'age': age,
    'watch_hours_week': watch_hours_week,
    'content_completion': content_completion,
    'device_type': device_type,
    'promo_used': promo_used,
    'support_tickets': support_tickets,
    'friend_referrals': friend_referrals,
    'subscription_renew': subscription_renew
})

print(f"Соотношение таргета: {df1['subscription_renew'].mean():.2f}")

df = df1.copy()

X = df.drop(columns = ['subscription_renew'])
y = df['subscription_renew']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = KNeighborsClassifier()
model.fit(X_train, y_train)

y_test_pred = model.predict(X_test)

print('')
print('До локализаций')
print('')
accuracy = accuracy_score(y_test, y_test_pred)
print('accuracy =', accuracy)
precision = precision_score(y_test, y_test_pred)
print('precision =', precision)
recall = recall_score(y_test, y_test_pred)
print('recall =', recall)
f1 = f1_score(y_test, y_test_pred)
print('f1 =', f1)

X = df.drop(columns = ['subscription_renew'])
y = df['subscription_renew']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train_0 = X_train.loc[X_train['tariff_type'] == 0]
y_train_0 = y_train.loc[X_train_0.index]
model_0 = KNeighborsClassifier()
model_0.fit(X_train_0, y_train_0)

X_train_1 = X_train.loc[X_train['tariff_type'] == 1]
y_train_1 = y_train.loc[X_train_1.index]
model_1 = KNeighborsClassifier()
model_1.fit(X_train_1, y_train_1)

X_train_2 = X_train.loc[X_train['tariff_type'] == 2]
y_train_2 = y_train.loc[X_train_2.index]
model_2 = KNeighborsClassifier()
model_2.fit(X_train_2, y_train_2)

print('')
print('')
print('Частные скоры на классах')
print('')
X_test_0 = X_test.loc[X_test['tariff_type'] == 0]
y_test_0 = y_test.loc[X_test_0.index]
y_test_pred_0 = model_0.predict(X_test_0)
print('accuracy_0 =', accuracy_score(y_test_0, y_test_pred_0))
print('precision_0 =', precision_score(y_test_0, y_test_pred_0))
print('recall_0 =', recall_score(y_test_0, y_test_pred_0))
print('f1_0 =', f1_score(y_test_0, y_test_pred_0))
print('')
X_test_1 = X_test.loc[X_test['tariff_type'] == 1]
y_test_1 = y_test.loc[X_test_1.index]
y_test_pred_1 = model_1.predict(X_test_1)
print('accuracy_1 =', accuracy_score(y_test_1, y_test_pred_1))
print('precision_1 =', precision_score(y_test_1, y_test_pred_1))
print('recall_1 =', recall_score(y_test_1, y_test_pred_1))
print('f1_1 =', f1_score(y_test_1, y_test_pred_1))
print('')
X_test_2 = X_test.loc[X_test['tariff_type'] == 2]
y_test_2 = y_test.loc[X_test_2.index]
y_test_pred_2 = model_2.predict(X_test_2)
print('accuracy_2 =', accuracy_score(y_test_2, y_test_pred_2))
print('precision_2 =', precision_score(y_test_2, y_test_pred_2))
print('recall_2 =', recall_score(y_test_2, y_test_pred_2))
print('f1_2 =', f1_score(y_test_2, y_test_pred_2))

df_y_test_y_pred_0 = pd.DataFrame()
df_y_test_y_pred_0['y_test'] = y_test_0
df_y_test_y_pred_0['y_test_pred'] = y_test_pred_0
df_y_test_y_pred_0

df_y_test_y_pred_1 = pd.DataFrame()
df_y_test_y_pred_1['y_test'] = y_test_1
df_y_test_y_pred_1['y_test_pred'] = y_test_pred_1
df_y_test_y_pred_1

df_y_test_y_pred_2 = pd.DataFrame()
df_y_test_y_pred_2['y_test'] = y_test_2
df_y_test_y_pred_2['y_test_pred'] = y_test_pred_2
df_y_test_y_pred_2


df_y_test_y_pred = pd.concat([df_y_test_y_pred_0, df_y_test_y_pred_1, df_y_test_y_pred_2])
df_y_test_y_pred

print('')
print('')
print('Общие скоры после локализаций')
print('')
accuracy = accuracy_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('accuracy = ', accuracy)
precision = precision_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('precision = ', precision)
recall = recall_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('recall = ', recall)
f1 = f1_score(df_y_test_y_pred['y_test'], df_y_test_y_pred['y_test_pred'])
print('f1 = ', f1)


Соотношение таргета: 0.24

До локализаций

accuracy = 0.82
precision = 0.6829268292682927
recall = 0.4666666666666667
f1 = 0.5544554455445545


Частные скоры на классах

accuracy_0 = 0.9794871794871794
precision_0 = 1.0
recall_0 = 0.9310344827586207
f1_0 = 0.9642857142857143

accuracy_1 = 0.9741935483870968
precision_1 = 0.9411764705882353
recall_1 = 0.9411764705882353
f1_1 = 0.9411764705882353

accuracy_2 = 0.9866666666666667
precision_2 = 0.9333333333333333
recall_2 = 1.0
f1_2 = 0.9655172413793104


Общие скоры после локализаций

accuracy =  0.98
precision =  0.9661016949152542
recall =  0.95
f1 =  0.957983193277311


In [75]:
df1['watch_hours_week'].value_counts()

15.7    14
1.6     13
26.5    12
2.7     12
32.2    12
        ..
14.6     1
28.5     1
0.1      1
12.2     1
25.6     1
Name: watch_hours_week, Length: 396, dtype: int64